# 🚀 Модуль 3. Урок 3. Слияния датасетов (merge, concat) и продвинутая агрегация .groupby().agg()

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_03_python/lesson_03_pandas_aggregations_merge.ipynb)

## 📌 Бизнес-контекст и методология
В Pandas слияния таблиц выполняются через `pd.merge()`, а вычисление бизнес-метрик — через `.groupby().agg()`.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

df_orders = pd.read_csv(orders_csv)
df_cust = pd.read_csv(customers_csv)

df_full = df_orders.merge(df_cust, on='customer_id', how='inner')

city_metrics = df_full.groupby('city').agg(
    total_orders=('order_id', 'count'),
    unique_buyers=('customer_id', 'nunique')
).reset_index()

city_metrics['orders_per_buyer'] = city_metrics['total_orders'] / city_metrics['unique_buyers']
city_metrics = city_metrics.sort_values(by='total_orders', ascending=False)
city_metrics.head(5)


,city,total_orders,unique_buyers,orders_per_buyer
2,Nizhny Novgorod,348,139,2.503597
0,Kazan,344,144,2.388889
6,Yekaterinburg,332,143,2.321678
5,Samara,331,153,2.163399
1,Moscow,328,126,2.603175


### 🎯 Практическое задание 1: Анализ каналов привлечения
Рассчитайте для каждого канала (`acquisition_channel`) общее число заказов и число уникальных покупателей.

In [3]:
### ВАШ КОД ЗДЕСЬ
channel_metrics = df_full.groupby('acquisition_channel').agg(
    total_orders=('order_id', 'count'),
    unique_buyers=('customer_id', 'nunique')
).reset_index()

print(channel_metrics)

assert not channel_metrics.empty, "Датафрейм не должен быть пустым"
assert 'acquisition_channel' in channel_metrics.columns, "Колонка обязательна"
print("✅ Тест успешно пройден!")


  acquisition_channel  total_orders  unique_buyers
0              Direct           361            154
1    Email Newsletter           407            170
2      Organic Search           403            165
3            Paid Ads           393            155
4            Referral           379            165
5        Social Media           339            159
✅ Тест успешно пройден!


### 💼 Именованная агрегация
Используйте понятный синтаксис кортежей в `agg`:
```python
df.groupby('channel').agg(orders=('id', 'count'), users=('user_id', 'nunique'))
```